In [2]:
import numpy as np
import pandas as pd
import emcee
import corner
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Constants
M_sun = 2e33             # Solar mass in g
M_tot = 0.01 * M_sun     # Total mass of the ejecta in g
c = 3e10                 # Speed of light in cm/s
v_min = 0.1 * c          # Minimum velocity of massive shell in cm/s
beta = 3                 # factor for homologous expansion - ensures shells never overlap 
numshells = 10           # Number of shells

# Time array
t = np.logspace(0, 6, 50)              
# Velocity array for each shell
v_shells = np.linspace(0.1, 1, numshells) * c

# Mass distribution
M_shells = beta * M_tot * (v_min**(beta)) * (v_shells**(-beta-1))    # Mass fraction based on velocity
M_shells = M_shells * M_tot / np.sum(M_shells)                       # Normalised so the shells sum to M_0
E_int_shells = 0.5 * M_shells * (v_shells**2)                        # Internal energy per shell, assuming kinetic energy is similar to internal energy

# Dataframe of properties
shell_data = pd.DataFrame({'Velocity (cm/s)': v_shells, 'Mass (g)': M_shells, 'Internal Energy (erg)': E_int_shells})

# Constants associated with heating and radiative loss terms
av = 0.56
bv = 0.71
cv = 0.74
xr = 0.6
kv = [20, 5, 1]                  # Opacity ranges for kilonova colors

# Heating efficiency as a function of time
def eps_th_v(t):
    t_day = t / 86400  # Convert seconds to days
    exp_term = np.exp(-av * t_day)
    log_term = np.log(1 + 2 * bv * t_day**cv)
    result = 0.36 * (exp_term + (log_term / (2 * bv * t_day**cv)))
    return result

# R-process energy deposition rate
def e_dot_r(t):
    t0 = 1.3  
    sigma = 0.11  
    arctan_term = np.arctan((t - t0) / sigma)
    result = 4e18 * eps_th_v(t) * (0.5 - (1 / np.pi) * arctan_term)**1.3
    return result

# Heating due to radioactive decay
def Q_rv(Mv, t):
    return Mv * xr * e_dot_r(t)

# Diffusion time
def t_diff(M_shells, t):
    return ((M_shells**(4./3.)) * kv[2]) / (4 * np.pi * (M_tot**(1./3.)) * v_min * t * c)

# Radiative losses
def L_v(R, Eint, t_diff):
    t_lc = (R / c)                            # Light-crossing time
    return Eint / (t_diff + t_lc)

# ODEs governing energy and dynamics
def energy(y, t, M):
    Eint, R, v = y
    diff_time = t_diff(M, t)
    dEint = -((Eint / R) * v) + Q_rv(M, t) - L_v(R, Eint, diff_time)
    dR = v
    dv = Eint / (M * R)
    return [dEint, dR, dv]

# Solving for each shell
E_all = []
R_all = []
v_all = []
for i, row in shell_data.iterrows():
    M = row['Mass (g)']
    Eint0 = row['Internal Energy (erg)']
    v0 = row['Velocity (cm/s)']
    R0 = 1e9  # Initial radius
    y0 = [ Eint0, R0, v0]
    Edep = odeint(energy, y0, t, args=(M,))
    E_all.append(Edep[:, 0])  
    R_all.append(Edep[:, 1])  
    v_all.append(Edep[:, 2])

# Compute photosphere radius
def compute_photosphere_radius(R_all, shell_masses, t):
    num_shells, num_times = R_all.shape
    R_ph = np.zeros(num_times)
    for i in range(num_times):
        tau_cumu = 0.0
        tau_list = []
        R_list = []
        for j in range(num_shells - 1, -1, -1):
            R_curr = R_all[j, i]
            d_tau = kv[2] * shell_masses[j] / (4 * np.pi * R_curr**2)
            tau_cumu += d_tau
            tau_list.append(tau_cumu)
            R_list.append(R_curr)
        tau_array = np.array(tau_list)
        R_array = np.array(R_list)
        if tau_array[-1] < 1:
            R_ph[i] = R_array[-1]
        else:
            indices = np.where(tau_array >= 1)[0]
            idx = indices[0]
            if idx == 0:
                R_ph[i] = R_array[0]
            else:
                tau_low = tau_array[idx - 1]
                tau_high = tau_array[idx]
                R_low = R_array[idx - 1]
                R_high = R_array[idx]
                f = (1 - tau_low) / (tau_high - tau_low)
                R_ph[i] = R_low + f * (R_high - R_low)
    return R_ph

R_ph = compute_photosphere_radius(np.array(R_all), M_shells, t)

L_tot = np.sum([L_v(R, Eint, t_diff(shell_data['Mass (g)'][i], t)) 
                for i, (R, Eint) in enumerate(zip(R_all, E_all))], axis=0)

sigma = 5.67e-5    # Stefan's constant in erg / cm^2 / s / K^4
h = 6.626e-27      # Planck's constant in erg * s
nu = 5e15          # Frequency (Hz), can be altered to see fluxes of different frequencies
kB = 1.38e-16      # Boltzmann's constant in erg / K
D = 1e25           # Luminosity distance in cm, as used by Metzger in his paper (100 Mpc)

T_eff = []
for i in range(len(R_ph)):
    T_eff_shell = (L_tot[i] / (4 * np.pi * sigma * (R_ph[i]**2)))**(1/4)
    T_eff.append(T_eff_shell)

Flux = []
for i in range(len(T_eff)):
    Flux_shell = ((2 * np.pi * h * (nu**3)) / (c**2)) * (1 / (np.exp((h * nu) / (kB * T_eff[i])) - 1)) * (R_ph[i]**2 / D**2)
    Flux.append(Flux_shell)  # Append the flux for each shell

Flux = np.array(Flux)

noise = 0.1 * Flux  # Add noise
Flux_perturbed = Flux + noise * np.random.randn(*Flux.shape)

def model(theta, t):
    M_tot, Eint0, R0, v0 = theta

    E_all, R_all, v_all = [], [], []
    
    for i, row in shell_data.iterrows():
        M = row['Mass (g)']
        y0 = [Eint0, R0, v0]  
        Edep = odeint(energy, y0, t, args=(M,))
        E_all.append(Edep[:, 0])  
        R_all.append(Edep[:, 1])  
        v_all.append(Edep[:, 2])

    R_ph = compute_photosphere_radius(np.array(R_all), M_shells, t)
    L_tot = np.sum([L_v(R, Eint, t_diff(shell_data['Mass (g)'][i], t)) 
                    for i, (R, Eint) in enumerate(zip(R_all, E_all))], axis=0)

    # Compute flux using updated L_tot
    T_eff = (L_tot / (4 * np.pi * sigma * (R_ph**2)))**(1/4)

    exponent = (h * nu) / (kB * T_eff)
    Flux_model = ((2 * np.pi * h * (nu**3)) / (c**2)) * (1 / (np.exp(exponent) - 1)) * (R_ph**2 / D**2)
    return Flux_model

def log_likelihood(theta, t, y, yerr):

    model_flux = model(theta, t)

    Loglike = -0.5 * np.sum(((y - model_flux) / yerr)**2)
    return Loglike


def check_initial_conditions(theta):                                  #Test initial conditions are being met. Unhash for use
    M_tot, Eint0, R0, v0 = theta
    if not (1e30 < M_tot < M_sun):
        raise ValueError(f"Invalid initial mass: {M_tot}.")
    if not (1e40 < Eint0 < 1e51):
        raise ValueError(f"Invalid initial energy: {Eint0}.")
    if not (1e8 < R0 < 1e17):
        raise ValueError(f"Invalid initial radius: {R0}.")
    if not (1e8 < v0 < 1e11):
        raise ValueError(f"Invalid initial velocity: {v0}.")

# Assign priors and reasonable ranges
def log_prior(theta):
    M_tot, Eint0, R0, v0 = theta
    if (1e30 < M_tot < M_sun and     # Mass range
        1e40 < Eint0 < 1e51 and      # Internal energy range
        1e8 < R0 < 1e17 and          # Initial radius range
        1e8 < v0 < 1e11):            # Initial velocity range
        return 0.0                   # Uniform prior
    return -np.inf  

def log_posterior(theta, t, y, yerr):
    lp = log_prior(theta)
    if np.isinf(lp):  
        return lp
    return lp + log_likelihood(theta, t, y, yerr)  

Eint0 = np.mean(E_int_shells)  
v0 = np.mean(v_shells)    
R0 = 1e9 

ndim = 4
nwalkers = 50
nsteps = 10000

mass_noise_scale = 0.1  
Eint0_noise_scale = 0.1  
v0_noise_scale = 0.05  
R0_noise_scale = 0.05

initial = np.array([M_tot, Eint0, R0, v0]) + np.array([mass_noise_scale * M_tot, Eint0_noise_scale * Eint0, 
                                                       R0_noise_scale * R0, v0_noise_scale * v0]) * np.random.randn(nwalkers, ndim)

# Ensure all values are physically meaningful (no negatives)
initial = np.abs(initial)

#check_initial_conditions(initial[0])           # Test function. Unhash for use

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior, args=(t, Flux_perturbed, noise))
sampler.run_mcmc(initial, nsteps, progress=True)

# Extract samples after burn-in (discard first 1000 steps)
samples = sampler.get_chain(discard=1000, thin=10, flat=True)

# Corner plot
param_range = [(np.min(samples[:, i]), np.max(samples[:, i])) for i in range(ndim)]

figure = corner.corner(samples, labels=["M_tot", "Eint0", "R0", "v0"], 
                       truths=[M_tot, Eint0, R0, v0], range=param_range)
plt.show()
samples = sampler.flatchain
samples[np.argmax(sampler.flatlnprobability)]
best_fit_params = samples[np.argmax(sampler.flatlnprobability)]
print("Best-fit parameters:", best_fit_params)

C:\Users\olibr\AppData\Local\Temp\ipykernel_15928\721423895.py:168: RuntimeWarning: overflow encountered in exp
  Flux_model = ((2 * np.pi * h * (nu**3)) / (c**2)) * (1 / (np.exp(exponent) - 1)) * (R_ph**2 / D**2)
  0%|          | 0/10000 [00:00<?, ?it/s]Traceback (most recent call last):
  File "c:\Users\olibr\pythonconda\Lib\site-packages\emcee\ensemble.py", line 640, in __call__
    return self.f(x, *self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\olibr\AppData\Local\Temp\ipykernel_15928\721423895.py", line 204, in log_posterior
    return lp + log_likelihood(theta, t, y, yerr)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\olibr\AppData\Local\Temp\ipykernel_15928\721423895.py", line 173, in log_likelihood
    model_flux = model(theta, t)
                 ^^^^^^^^^^^^^^^
  File "C:\Users\olibr\AppData\Local\Temp\ipykernel_15928\721423895.py", line 155, in model
    Edep = odeint(energy, y0, t, args=(M,))
           ^^^^^^^

emcee: Exception while calling your likelihood function:
  params: [1.97191249e+31 1.50767272e+49 9.38554187e+08 1.78309783e+10]
  args: (array([1.00000000e+00, 1.32571137e+00, 1.75751062e+00, 2.32995181e+00,
       3.08884360e+00, 4.09491506e+00, 5.42867544e+00, 7.19685673e+00,
       9.54095476e+00, 1.26485522e+01, 1.67683294e+01, 2.22299648e+01,
       2.94705170e+01, 3.90693994e+01, 5.17947468e+01, 6.86648845e+01,
       9.10298178e+01, 1.20679264e+02, 1.59985872e+02, 2.12095089e+02,
       2.81176870e+02, 3.72759372e+02, 4.94171336e+02, 6.55128557e+02,
       8.68511374e+02, 1.15139540e+03, 1.52641797e+03, 2.02358965e+03,
       2.68269580e+03, 3.55648031e+03, 4.71486636e+03, 6.25055193e+03,
       8.28642773e+03, 1.09854114e+04, 1.45634848e+04, 1.93069773e+04,
       2.55954792e+04, 3.39322177e+04, 4.49843267e+04, 5.96362332e+04,
       7.90604321e+04, 1.04811313e+05, 1.38949549e+05, 1.84206997e+05,
       2.44205309e+05, 3.23745754e+05, 4.29193426e+05, 5.68986603e+05,
       7.5

KeyboardInterrupt: 

In [1]:
y0 = [M_tot, Eint0, R0, v0]
plt.figure(figsize=(8, 6))
#plt.loglog(t, Flux, label='Total Flux')  # Plot the total flux (sum of all shells)
#plt.loglog(t, Flux_perturbed, label='Total Flux')  # Plot the total flux (sum of all shells)
#plt.errorbar(t, Flux, yerr = noise)
samples = sampler.flatchain
plt.loglog(t, model(y0, t), color="b")
for theta in samples[np.random.randint(len(samples), size=100)]:
    plt.plot(t, model(theta, t), color="r", alpha=0.05)
plt.xlabel('Time (s)')
plt.ylabel('Flux (erg/cm^2/s)')
plt.xlim([0, 1e6])
plt.ylim([1e-27, 1e-24])
plt.title('Flux Evolution over Time')
plt.grid(True)
plt.legend()
plt.show()

NameError: name 'M_tot' is not defined

In [35]:
import arviz as az
tau = sampler.get_autocorr_time()
print("Autocorrelation Time:", tau)
print("Mean Acceptance Fraction:", np.mean(sampler.acceptance_fraction))
r_hat = az.rhat(samples)
print("Gelman-Rubin R-hat values:", r_hat)

AutocorrError: The chain is shorter than 50 times the integrated autocorrelation time for 4 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 2;
tau: [ 8.46197936 11.94349696  8.47906279  9.89692496]

In [36]:
print(np.isnan(sampler.chain).sum())  # Count NaN values
print(np.isinf(sampler.chain).sum())  # Count Inf values
print(sampler.chain.shape)  # Check the size of the chain (samples, parameters, steps)


0
0
(100, 100, 4)


In [44]:
print(np.var(sampler.chain[:, :, 3]))  # Variance of the first parameter

1.707972407038489e+19


In [ ]:
import matplotlib.pyplot as plt
plt.plot(sampler.chain[:, :, 0])  # Trace plot of the first parameter
plt.show()

NameError: name 'sampler' is not defined